# Chapter 17: Statistical Learning Theory and Regularization. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/17-theoryai.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/17-theoryai.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/17-theoryai.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("boot", "gamlr", "glmnet", "leaps", "MASS", "textir")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 17.9: Baseball Batting Averages

Efron and Morris (1975) analyzed the batting averages of 18 baseball players based on their first 45 at-bats. This exercise uses the grand-mean-centered James-Stein estimator, which shrinks individual averages toward the grand mean $\bar y$:
$$
\hat{\theta}_i^{JS} = \bar y + \left(1 - \frac{(p-3)\sigma^2}{\sum_j (y_j - \bar y)^2}\right)(y_i - \bar y)
$$
This is a different, deliberate variant from the zero-centered estimator in Pen-and-Pencil Exercise 1(b), which shrinks toward the origin using a $(p-2)$ correction instead.

a) Simulate the scenario: generate true season batting averages $\theta_i \sim N(0.265, 0.02^2)$ for 18 players. For each player, generate $y_i \sim N(\theta_i, \sigma^2)$ where $\sigma^2 = 0.265 \times 0.735 / 45$ (binomial variance approximation).

b) Compute the MLE ($\hat{\theta}_i = y_i$) and the James-Stein estimator above. Plot both estimates alongside the true $\theta_i$.

c) Compute the total squared error $\sum_i (\hat{\theta}_i - \theta_i)^2$ for both estimators. Which is smaller?

d) Repeat the simulation 1000 times and compute the average total squared error for MLE vs. JS. Report the risk ratio.

## Exercise 17.10: Ridge and Lasso Comparison

Use the `Boston` dataset (`MASS` package) to compare Ridge and Lasso regression.

a) Standardize all predictors. Fit Ridge regression for a grid of 50 $\lambda$ values from $10^{-2}$ to $10^4$ (log-spaced). Plot the coefficient paths (all 13 coefficients vs. $\log \lambda$).

b) Repeat for Lasso. How do the coefficient paths differ? Which coefficients are driven to exactly zero first?

c) Use 10-fold cross-validation to select the optimal $\lambda$ for both Ridge and Lasso. Report the selected $\lambda$ and test MSE.

d) Compare the number of nonzero coefficients at the optimal $\lambda$ for each method.

## Exercise 17.11: Single Best Replacement for Subset Selection

The Single Best Replacement (SBR) algorithm attacks the $\ell_0$ objective $f(S) = \tfrac12\lVert y - X_S\hat\beta_S\rVert_2^2 + \lambda|S|$ directly, one add-or-remove move at a time. This exercise runs it on the `Hitters` data (`data/Hitters.csv`), the classic subset-selection benchmark of major-league batters (1986-season and career statistics predicting `Salary`). Drop rows with missing `Salary`, set $y = \log(\text{Salary})$ centered to mean zero, and standardize the 16 numeric predictors `AtBat, Hits, HmRun, Runs, RBI, Walks, Years, CAtBat, CHits, CHmRun, CRuns, CRBI, CWalks, PutOuts, Assists, Errors` (so $n=263$, $p=16$; with $y$ centered and $X$ standardized, no separate intercept is needed).

a) Implement SBR as described in the chapter: start from the empty active set $S_0=\emptyset$; at each iteration, for every $j=1,\ldots,p$ evaluate $f(S\cdot j)$, where $S\cdot j$ adds $j$ if $j\notin S$ and removes it if $j\in S$ and $\hat\beta_S$ is the OLS fit of $y$ on $X_S$; accept the best replacement if it lowers $f$, otherwise stop. (Direct refitting each time is fine at this scale.) Run this for $\lambda\in\{1,3,12\}$ and report, for each $\lambda$, the final active set, the number of iterations to convergence, and a plot of $f(S_k)$ against $k$.

b) For each $\lambda$ in part (a), find the `glmnet` Lasso solution with the same number of nonzero coefficients as SBR's active set (search the Lasso path for a matching model size). Do SBR and Lasso agree on which predictors they select?

c) Since $p=16$, exhaustive best-subset selection is feasible. Use `leaps::regsubsets` to get the best subset of each size $k=1,\ldots,6$ by residual sum of squares (RSS). For the active-set size found by SBR at $\lambda=3$, compare SBR's set and RSS to the true best subset of the same size. Does SBR find the global optimum?

d) Build a warm-start active set from three steps of pure forward selection (ignore the penalty; just add whichever variable most reduces RSS, three times). Rerun SBR at $\lambda=3$ from this warm start. Does it converge to the same final active set as the empty start? In how many iterations? What does this say about SBR's guarantee, or lack of one, of global optimality?

## Exercise 17.12: Credit Scoring with LASSO

Use the German credit data (`credit.csv`) to predict loan default using regularized logistic regression.

a) Fit a standard logistic regression predicting `Default` from loan characteristics (duration, amount, installment, age, history). Interpret key coefficients.

b) Fit a LASSO logistic regression using `cv.gamlr`. Compare the selected variables to the unregularized model.

c) Compare 10-fold CV error for both models. Which performs better out-of-sample?

d) Discuss: When would LASSO provide significant improvement over standard logistic regression?

## Exercise 17.13: Congressional Speech Analysis

Use the `congress109` dataset from the `textir` package containing word counts from speeches of the 109th Congress.

a) Load `congress109Counts` and `congress109Ideology`. What is the dimension of the counts matrix?

b) Use LASSO to regress `repshare` (proportion of Republican votes) on binarized word presence (>0). Which words are most associated with each party?

c) Perform 5-fold cross-validation. How well can we predict partisan lean from word usage?

d) Discuss: Why might predicting partisanship from speech be easier (or harder) than predicting policy positions?